# Rodar todos os modelos do grupo

Este notebook roda os 5 modelos (goal programming, linear Sharpe MF/Piotroski, mínima variância MF/Piotroski) com **os mesmos hiperparâmetros** e salva cada carteira na pasta certa.

**Para trocar de grupo:** altere só a célula de hiperparâmetros e rode tudo (*Run All*).

In [1]:
import os
import json
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import SolverFactory, SolverStatus, TerminationCondition

## Hiperparâmetros do grupo (a única célula que você precisa mudar)

In [2]:
PESO_MAXIMO = 0.10          # peso máximo por ativo
CARDINALIDADE_MAX = 20      # número máximo de ativos | None = sem limite
THETA = 0.75                # linear Sharpe: peso do retorno | min var: peso da variância
META_SCORE_GP = 0.9         # meta dos scores no goal programming

# Modelos a rodar (comente uma linha para pular um modelo)
MODELOS = [
    'goal_programming',
    'linear_sharpe_mf',
    'linear_sharpe_piotroski',
    'min_var_mf',
    'min_var_piotroski',
]

## Caminhos e configurações gerais

In [3]:
PASTA_RETORNOS = '../../../novos-dados-retornos/otimizacao/'
PASTA_SCORES = '../../../score-raspagem/'
PASTA_CARTEIRAS = '../carteiras/'

# PC
CPLEX_PATH = r'C:\Program Files\IBM\ILOG\CPLEX_Studio_Community222\cplex\bin\x64_win64\cplex.exe'
# NOTEBOOK
# CPLEX_PATH = r'C:\CPLEX_Studio2211\cplex\bin\x64_win64\cplex.exe'

ANOS = list(range(2016, 2027))   # carteira do ano X: score de X-1, retornos jan-mar de X
PESO_RESIDUO = 5e-5              # pesos menores que isso são resíduo numérico do solver
KMAX = 100                       # big-M da linearização do Sharpe

# onde cada modelo salva a carteira (mesmos nomes de antes, o backtest continua funcionando)
ARQUIVOS = {
    'goal_programming':        'goal_programming/carteiras_goal_programming.csv',
    'linear_sharpe_mf':        'linear_sharpe_mf/carteiras_sharpe_linear_magic_formula.csv',
    'linear_sharpe_piotroski': 'linear_sharpe_piotroski/carteiras_sharpe_linear_piotroski.csv',
    'min_var_mf':              'min_var_mf/carteiras_minvar_magic_formula.csv',
    'min_var_piotroski':       'min_var_piotroski/carteiras_minvar_piotroski.csv',
}

opt = SolverFactory('cplex', executable=CPLEX_PATH)

## Leitura dos dados (uma vez só, para todos os modelos)

In [4]:
def ler_retornos(ano):
    df = pd.read_csv(f'{PASTA_RETORNOS}df_ativos_{ano}.csv', parse_dates=['date']).set_index('date')
    df = df.loc[:, ~(df == 0).all()]          # remove ativos sem nenhum dado
    prop_zeros = (df == 0).mean()
    return df.loc[:, prop_zeros < 0.5]        # remove ativos com 50% ou mais de dias zerados


def ler_score(tipo, ano_score):
    if tipo == 'mf':
        arquivo, coluna = f'{PASTA_SCORES}magic_formula/resultado/{ano_score}.xlsx', 'magic_formula'
    else:
        arquivo, coluna = f'{PASTA_SCORES}piotroski/resultado/{ano_score}.xlsx', 'p_norm'
    s = pd.read_excel(arquivo).set_index('Unnamed: 0')[coluna]
    return s.dropna()                         # Series: ticker -> score


retornos = {ano: ler_retornos(ano) for ano in ANOS}
scores = {(tipo, ano): ler_score(tipo, ano - 1) for tipo in ['mf', 'pio'] for ano in ANOS}

for ano in ANOS:
    print(f"{ano}: {retornos[ano].shape[1]} ativos | {retornos[ano].index[0].date()} a {retornos[ano].index[-1].date()}")

2016: 68 ativos | 2016-01-05 a 2016-03-31
2017: 69 ativos | 2017-01-03 a 2017-03-31
2018: 75 ativos | 2018-01-03 a 2018-03-29
2019: 77 ativos | 2019-01-03 a 2019-03-29
2020: 80 ativos | 2020-01-03 a 2020-03-31
2021: 90 ativos | 2021-01-05 a 2021-03-31
2022: 98 ativos | 2022-01-04 a 2022-03-31
2023: 99 ativos | 2023-01-03 a 2023-03-31
2024: 99 ativos | 2024-01-03 a 2024-03-28
2025: 99 ativos | 2025-01-03 a 2025-03-31
2026: 100 ativos | 2026-01-05 a 2026-03-31


## Funções auxiliares

In [5]:
def normalizar_retornos(ret):
    """Min-max por ativo, como nos notebooks originais. Remove colunas constantes (evita 0/0)."""
    amplitude = ret.max() - ret.min()
    ret = ret.loc[:, amplitude > 0]
    return (ret - ret.min()) / (ret.max() - ret.min())


def resolver(model):
    """Resolve e para com erro se o solver não chegar no ótimo."""
    res = opt.solve(model, tee=False)
    tc = res.solver.termination_condition
    if not (res.solver.status == SolverStatus.ok and tc == TerminationCondition.optimal):
        raise RuntimeError(f'solver terminou com {tc}')


def restricao_cardinalidade(model):
    """Só cria a restrição se CARDINALIDADE_MAX não for None."""
    if CARDINALIDADE_MAX is not None:
        model.r_card_max = pyo.Constraint(expr=sum(model.y[a] for a in model.A) <= CARDINALIDADE_MAX)

## Modelos

### Sharpe linearizado (MF e Piotroski)
Igual aos notebooks originais, com o retorno **médio** na função objetivo (soma dos dias ÷ número de dias). O retorno médio já é calculado antes, o que dá o mesmo resultado e monta o modelo mais rápido.

In [6]:
def modelo_linear_sharpe(ret, score):
    ativos = ret.columns.intersection(score.index)
    r = normalizar_retornos(ret[ativos])
    ativos = r.columns
    ret_medio = r.mean()          # = soma dos dias / número de dias
    cov = r.cov()
    s = score[ativos]
    n = len(ativos)

    m = pyo.ConcreteModel()
    m.A = pyo.RangeSet(0, n - 1)
    m.mu = pyo.Param(m.A, initialize=lambda m, a: float(ret_medio.iloc[a]))
    m.sigma = pyo.Param(m.A, m.A, initialize=lambda m, a, b: float(cov.iloc[a, b]))
    m.score = pyo.Param(m.A, initialize=lambda m, a: float(s.iloc[a]))

    # z = k*x (linearização do Sharpe)
    m.z = pyo.Var(m.A, domain=pyo.NonNegativeReals)
    m.k = pyo.Var(domain=pyo.NonNegativeReals)
    m.y = pyo.Var(m.A, domain=pyo.Binary)
    m.w = pyo.Var(m.A, domain=pyo.NonNegativeReals)   # w = k*y

    m.r_y_zero = pyo.Constraint(m.A, rule=lambda m, a: m.w[a] <= KMAX * m.y[a])
    m.r_teto = pyo.Constraint(m.A, rule=lambda m, a: m.w[a] <= m.k)
    m.r_limite = pyo.Constraint(m.A, rule=lambda m, a: m.w[a] >= m.k - KMAX * (1 - m.y[a]))
    m.r_pesomax = pyo.Constraint(m.A, rule=lambda m, a: m.z[a] <= PESO_MAXIMO * m.w[a])
    restricao_cardinalidade(m)
    m.r_soma_z = pyo.Constraint(expr=sum(m.z[a] for a in m.A) == m.k)
    m.r_risco = pyo.Constraint(expr=sum(m.sigma[a, b] * m.z[a] * m.z[b] for a in m.A for b in m.A) <= 1)

    retorno = sum(m.z[a] * m.mu[a] for a in m.A)
    score_cart = sum(m.z[a] * m.score[a] for a in m.A)
    m.obj = pyo.Objective(expr=THETA * retorno + (1 - THETA) * score_cart, sense=pyo.maximize)

    resolver(m)
    k = pyo.value(m.k)
    return {ativos[a]: pyo.value(m.z[a]) / k for a in m.A}

### Mínima variância com score (MF e Piotroski)
Variância dividida pela variância da carteira igualmente ponderada (média de todos os elementos do Σ) e score dividido pelo score médio.

In [7]:
def modelo_min_var(ret, score):
    ativos = ret.columns.intersection(score.index)
    r = normalizar_retornos(ret[ativos])
    ativos = r.columns
    cov = r.cov()
    s = score[ativos]
    n = len(ativos)

    escala_var = float(cov.values.mean())    # variância da carteira igualmente ponderada
    escala_score = float(s.mean())           # score da carteira igualmente ponderada

    m = pyo.ConcreteModel()
    m.A = pyo.RangeSet(0, n - 1)
    m.sigma = pyo.Param(m.A, m.A, initialize=lambda m, a, b: float(cov.iloc[a, b]))
    m.score = pyo.Param(m.A, initialize=lambda m, a: float(s.iloc[a]))
    m.x = pyo.Var(m.A, bounds=(0, 1))
    m.y = pyo.Var(m.A, domain=pyo.Binary)

    m.r_pesomax = pyo.Constraint(m.A, rule=lambda m, a: m.x[a] <= PESO_MAXIMO * m.y[a])
    m.r_soma = pyo.Constraint(expr=sum(m.x[a] for a in m.A) == 1)
    restricao_cardinalidade(m)

    variancia = sum(m.x[a] * m.x[b] * m.sigma[a, b] for a in m.A for b in m.A)
    score_cart = sum(m.x[a] * m.score[a] for a in m.A)
    m.obj = pyo.Objective(expr=THETA * (variancia / escala_var) - (1 - THETA) * (score_cart / escala_score),
                          sense=pyo.minimize)

    resolver(m)
    return {ativos[a]: pyo.value(m.x[a]) for a in m.A}

### Goal programming
Metas para o score médio ponderado do Piotroski e do Magic Formula. Não usa θ.

In [9]:
def modelo_goal_programming(ret, score_mf, score_pio):
    ativos = ret.columns.intersection(score_pio.index).intersection(score_mf.index)
    s_mf = score_mf[ativos]
    s_pio = score_pio[ativos]
    n = len(ativos)

    m = pyo.ConcreteModel()
    m.A = pyo.RangeSet(0, n - 1)
    m.s_mf = pyo.Param(m.A, initialize=lambda m, a: float(s_mf.iloc[a]))
    m.s_pio = pyo.Param(m.A, initialize=lambda m, a: float(s_pio.iloc[a]))
    m.x = pyo.Var(m.A, bounds=(0, 1))
    m.y = pyo.Var(m.A, domain=pyo.Binary)
    m.d_pio_menos = pyo.Var(domain=pyo.NonNegativeReals)
    m.d_pio_mais = pyo.Var(domain=pyo.NonNegativeReals)
    m.d_mf_menos = pyo.Var(domain=pyo.NonNegativeReals)
    m.d_mf_mais = pyo.Var(domain=pyo.NonNegativeReals)

    m.r_meta_pio = pyo.Constraint(expr=sum(m.x[a] * m.s_pio[a] for a in m.A)
                                  + m.d_pio_menos - m.d_pio_mais >= META_SCORE_GP)
    m.r_meta_mf = pyo.Constraint(expr=sum(m.x[a] * m.s_mf[a] for a in m.A)
                                 + m.d_mf_menos - m.d_mf_mais >= META_SCORE_GP)
    m.r_pesomax = pyo.Constraint(m.A, rule=lambda m, a: m.x[a] <= PESO_MAXIMO * m.y[a])
    m.r_soma = pyo.Constraint(expr=sum(m.x[a] for a in m.A) == 1)
    restricao_cardinalidade(m)

    m.obj = pyo.Objective(expr=m.d_pio_menos + m.d_mf_menos, sense=pyo.minimize)

    resolver(m)
    return {ativos[a]: pyo.value(m.x[a]) for a in m.A}

## Rodar todos os modelos e salvar

In [10]:
FUNCOES = {
    'goal_programming':        lambda ano: modelo_goal_programming(retornos[ano], scores[('mf', ano)], scores[('pio', ano)]),
    'linear_sharpe_mf':        lambda ano: modelo_linear_sharpe(retornos[ano], scores[('mf', ano)]),
    'linear_sharpe_piotroski': lambda ano: modelo_linear_sharpe(retornos[ano], scores[('pio', ano)]),
    'min_var_mf':              lambda ano: modelo_min_var(retornos[ano], scores[('mf', ano)]),
    'min_var_piotroski':       lambda ano: modelo_min_var(retornos[ano], scores[('pio', ano)]),
}


def montar_df_portfolios(carteiras):
    """Tira só os resíduos numéricos e confere se cada carteira soma 1 e respeita o peso máximo."""
    linhas = [{'ano': ano, 'ativo': ativo, 'peso': peso}
              for ano, pesos in carteiras.items()
              for ativo, peso in pesos.items() if peso >= PESO_RESIDUO]
    df = pd.DataFrame(linhas)

    soma = df.groupby('ano')['peso'].sum()
    if ((soma - 1).abs() > 0.01).any():
        raise RuntimeError(f'Pesos não somam 1 nos anos:\n{soma[(soma - 1).abs() > 0.01]}')
    acima = df[df['peso'] > PESO_MAXIMO + 1e-4]
    if not acima.empty:
        raise RuntimeError(f'Pesos acima do máximo:\n{acima}')
    return df


resultados = {}
for nome in MODELOS:
    print(f'===== {nome} =====')
    carteiras = {}
    for ano in ANOS:
        try:
            carteiras[ano] = FUNCOES[nome](ano)
        except Exception as e:
            raise RuntimeError(f'{nome} | ano {ano}: {e}') from e
        n_ativos = sum(p >= PESO_RESIDUO for p in carteiras[ano].values())
        print(f'  {ano}: {n_ativos} ativos')

    df_portfolios = montar_df_portfolios(carteiras)
    caminho = os.path.join(PASTA_CARTEIRAS, ARQUIVOS[nome])
    os.makedirs(os.path.dirname(caminho), exist_ok=True)
    df_portfolios.to_csv(caminho)
    resultados[nome] = df_portfolios
    print(f'  salvo em {caminho}')

# registra com quais parâmetros essas carteiras foram geradas
parametros = {'PESO_MAXIMO': PESO_MAXIMO, 'CARDINALIDADE_MAX': CARDINALIDADE_MAX,
              'THETA': THETA, 'META_SCORE_GP': META_SCORE_GP, 'MODELOS': MODELOS}
with open(os.path.join(PASTA_CARTEIRAS, 'parametros.json'), 'w') as f:
    json.dump(parametros, f, indent=2)
print('\nParâmetros usados:', parametros)

===== goal_programming =====
  2016: 10 ativos
  2017: 10 ativos
  2018: 10 ativos
  2019: 10 ativos
  2020: 10 ativos
  2021: 10 ativos
  2022: 10 ativos
  2023: 10 ativos
  2024: 10 ativos
  2025: 10 ativos
  2026: 10 ativos
  salvo em ../carteiras/goal_programming/carteiras_goal_programming.csv
===== linear_sharpe_mf =====
  2016: 20 ativos
  2017: 20 ativos
  2018: 20 ativos
  2019: 17 ativos
  2020: 11 ativos
  2021: 17 ativos
  2022: 16 ativos
  2023: 20 ativos
  2024: 19 ativos
  2025: 16 ativos
  2026: 17 ativos
  salvo em ../carteiras/linear_sharpe_mf/carteiras_sharpe_linear_magic_formula.csv
===== linear_sharpe_piotroski =====
  2016: 17 ativos
  2017: 19 ativos
  2018: 20 ativos
  2019: 20 ativos
  2020: 13 ativos
  2021: 18 ativos
  2022: 15 ativos
  2023: 17 ativos
  2024: 19 ativos
  2025: 15 ativos
  2026: 14 ativos
  salvo em ../carteiras/linear_sharpe_piotroski/carteiras_sharpe_linear_piotroski.csv
===== min_var_mf =====
  2016: 20 ativos
  2017: 19 ativos
  2018: 20 a

## Conferência rápida

In [11]:
# número de ativos por ano em cada modelo
pd.DataFrame({nome: df.groupby('ano')['ativo'].count() for nome, df in resultados.items()})

,goal_programming,linear_sharpe_mf,linear_sharpe_piotroski,min_var_mf,min_var_piotroski
ano,,,,,
2016,10,20,17,20,18
2017,10,20,19,19,19
2018,10,20,20,20,19
2019,10,17,20,15,18
2020,10,11,13,12,14
2021,10,17,18,20,20
2022,10,16,15,16,19
2023,10,20,17,20,18
2024,10,19,19,20,20


In [12]:
# carteira de um modelo num ano específico
modelo, ano = 'min_var_mf', 2020
resultados[modelo][resultados[modelo]['ano'] == ano].sort_values('peso', ascending=False)

,ano,ativo,peso
77,2020,BPAC11,0.100000
76,2020,BBSE3,0.100000
85,2020,YDUQ3,0.100000
84,2020,VIVA3,0.100000
83,2020,SUZB3,0.100000
74,2020,ABEV3,0.100000
80,2020,RADL3,0.100000
81,2020,SAUD3,0.100000
82,2020,SBFG3,0.100000
78,2020,IRBR3,0.066727
